# 04 Visualize Metric Scores

Visual analysis for PR suggestion coverage metrics.

Use the ML-only virtual environment:

```bash
source ../../.venv/bin/activate
python -m ipykernel install --user --name pipeline-fl-ml --display-name "pipeline-fl-ml"
```

Then select the `pipeline-fl-ml` kernel in Jupyter.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib-cache'))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

SCORES_PATH = PROJECT_ROOT / 'data' / 'external' / 'github_codereview' / 'metric_scores.csv'
LABEL_ORDER = ['0%', 'partial', 'mostly', '100%']
PERCENTAGE_BUCKET_ORDER = ['0', '1-10', '11-20', '21-30', '31-40', '41-50', '51-60', '61-70', '71-80', '81-90', '91-100']

def percentage_bucket(value: int | float) -> str:
    percentage = max(0, min(100, int(round(value))))
    if percentage == 0:
        return '0'
    lower_bound = ((percentage - 1) // 10) * 10 + 1
    upper_bound = min(lower_bound + 9, 100)
    return f'{lower_bound}-{upper_bound}'

P1_LANGUAGES = ['jupyter_notebook', 'html', 'groovy', 'hcl', 'shell', 'typescript', 'c', 'javascript']
BASE_METRIC_COLUMNS = [
    'line_recall',
    'token_recall',
    'identifier_normalized_token_recall',
    'literal_normalized_token_recall',
    'identifier_and_literal_normalized_token_recall',
    'best_hunk_token_recall',
    'best_hunk_token_precision',
    'best_hunk_token_f1',
    'best_hunk_identifier_normalized_recall',
    'best_hunk_literal_normalized_recall',
    'best_hunk_identifier_and_literal_normalized_recall',
    'best_hunk_contiguous_line_ratio',
    'best_hunk_token_lcs_recall',
    'best_hunk_size_ratio',
    'meaningful_anchor_recall',
    'structural_similarity',
    'structural_node_recall',
    'changed_line_overlap_ratio',
]
COUNT_COLUMNS = [
    'meaningful_anchor_count',
    'best_hunk_size',
    'candidate_hunk_count',
]
GUMTREE_METRIC_COLUMNS = [
    'gumtree_insert_ratio',
    'gumtree_delete_ratio',
    'gumtree_update_ratio',
    'gumtree_move_ratio',
]

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)
SCORES_PATH


## Load Scores

In [ ]:
if not SCORES_PATH.exists():
    raise FileNotFoundError(f'Missing score file: {SCORES_PATH}. Run notebook 03_evaluate first.')

scores = pd.read_csv(SCORES_PATH)
scores['is_correct'] = scores['label'] == scores['predicted_label']
scores['absolute_error'] = (scores['expected_landed_percentage'] - scores['predicted_percentage']).abs()
if 'expected_percentage_bucket' not in scores.columns:
    scores['expected_percentage_bucket'] = scores['expected_landed_percentage'].apply(percentage_bucket)
if 'predicted_percentage_bucket' not in scores.columns:
    scores['predicted_percentage_bucket'] = scores['predicted_percentage'].apply(percentage_bucket)
scores['expected_percentage_bucket'] = pd.Categorical(scores['expected_percentage_bucket'], categories=PERCENTAGE_BUCKET_ORDER, ordered=True)
scores['predicted_percentage_bucket'] = pd.Categorical(scores['predicted_percentage_bucket'], categories=PERCENTAGE_BUCKET_ORDER, ordered=True)
scores['label'] = pd.Categorical(scores['label'], categories=LABEL_ORDER, ordered=True)
scores['predicted_label'] = pd.Categorical(scores['predicted_label'], categories=LABEL_ORDER, ordered=True)

for column in BASE_METRIC_COLUMNS + GUMTREE_METRIC_COLUMNS + COUNT_COLUMNS:
    if column in scores.columns:
        scores[column] = pd.to_numeric(scores[column], errors='coerce')

if 'gumtree_available' in scores.columns:
    scores['gumtree_available'] = scores['gumtree_available'].astype(str).str.lower().eq('true')
if 'structural_available' in scores.columns:
    scores['structural_available'] = scores['structural_available'].astype(str).str.lower().eq('true')

for column in ['suggestion_language', 'tokenizer']:
    if column not in scores.columns:
        scores[column] = 'unknown'
    scores[column] = scores[column].fillna('unknown').astype(str)

available_metric_columns = [column for column in BASE_METRIC_COLUMNS + GUMTREE_METRIC_COLUMNS if column in scores.columns]
scores.head()


## Dataset Summary

In [ ]:
summary = pd.DataFrame([
    {'metric': 'examples', 'value': len(scores)},
    {'metric': 'accuracy', 'value': round(scores['is_correct'].mean(), 3)},
    {'metric': 'mean_absolute_error', 'value': round(scores['absolute_error'].mean(), 2)},
    {'metric': 'mistakes', 'value': int((~scores['is_correct']).sum())},
])
summary

## Language And Tokenizer Coverage

This shows whether the evaluator is using language-specific handling or falling back to generic regex tokens.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

language_counts = scores['suggestion_language'].value_counts().sort_values(ascending=False)
tokenizer_counts = scores['tokenizer'].value_counts().sort_values(ascending=False)

sns.barplot(x=language_counts.index, y=language_counts.values, ax=axes[0], color='#4C78A8')
axes[0].set_title('Suggestion language')
axes[0].set_xlabel('language')
axes[0].set_ylabel('examples')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(x=tokenizer_counts.index, y=tokenizer_counts.values, ax=axes[1], color='#F58518')
axes[1].set_title('Tokenizer strategy')
axes[1].set_xlabel('tokenizer')
axes[1].set_ylabel('examples')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

pd.crosstab(scores['suggestion_language'], scores['tokenizer']).sort_index()


## Candidate Hunk Types

This shows which candidate source produced the best match: same file, neighboring hunks, same extension, or anchor overlap.


In [ ]:
if 'best_hunk_candidate_type' not in scores.columns:
    print('No candidate hunk columns found. Run notebook 03_evaluate again.')
else:
    candidate_counts = scores['best_hunk_candidate_type'].fillna('none').replace('', 'none').value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(x=candidate_counts.index, y=candidate_counts.values, ax=ax, color='#72B7B2')
    ax.set_title('Best candidate hunk type')
    ax.set_xlabel('candidate type')
    ax.set_ylabel('examples')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()

    pd.crosstab(scores['best_hunk_candidate_type'].fillna('none').replace('', 'none'), scores['predicted_label'])


## Structural Availability

This shows where AST/structural evidence is available. Python and Jupyter code cells use local `ast`; Go/C++/Rust/Java/TypeScript/JavaScript/C/HTML use local Tree-sitter when parser checks pass. GumTree remains separate and opt-in.


In [ ]:
if 'structural_available' not in scores.columns:
    print('No structural columns found. Run notebook 03_evaluate again.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    availability_counts = scores['structural_available'].value_counts().rename(index={True: 'available', False: 'unavailable'})
    sns.barplot(x=availability_counts.index, y=availability_counts.values, ax=axes[0], color='#54A24B')
    axes[0].set_title('Structural scoring availability')
    axes[0].set_xlabel('status')
    axes[0].set_ylabel('examples')

    engine_counts = scores['structural_engine'].fillna('').replace('', 'none').value_counts()
    sns.barplot(x=engine_counts.index, y=engine_counts.values, ax=axes[1], color='#B279A2')
    axes[1].set_title('Structural engine')
    axes[1].set_xlabel('engine')
    axes[1].set_ylabel('examples')
    axes[1].tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()

    pd.crosstab(scores['suggestion_language'], scores['structural_engine'].fillna('').replace('', 'none'))


## P1 Language Readiness

This focuses on the second language group: Jupyter Notebook, HTML, Groovy, HCL, Shell, TypeScript, C, and JavaScript.


In [ ]:
p1_scores = scores[scores['suggestion_language'].isin(P1_LANGUAGES)].copy()
if p1_scores.empty:
    print('No P1 examples in the current score file.')
else:
    readiness = (
        p1_scores.groupby('suggestion_language')
        .agg(
            examples=('example_id', 'count'),
            accuracy=('is_correct', 'mean'),
            mean_absolute_error=('absolute_error', 'mean'),
            structural_available=('structural_available', 'mean'),
        )
        .sort_values('examples', ascending=False)
    )
    display(readiness)
    display(pd.crosstab(p1_scores['suggestion_language'], p1_scores['tokenizer']))
    display(pd.crosstab(p1_scores['suggestion_language'], p1_scores['structural_engine'].fillna('').replace('', 'none')))


## Actual vs Predicted Label Distribution

In [ ]:
distribution = (
    pd.concat([
        scores['label'].value_counts().rename('actual'),
        scores['predicted_label'].value_counts().rename('predicted'),
    ], axis=1)
    .reindex(LABEL_ORDER)
    .fillna(0)
    .astype(int)
)

ax = distribution.plot(kind='bar', figsize=(8, 4), rot=0)
ax.set_title('Actual vs Predicted Label Counts')
ax.set_xlabel('label')
ax.set_ylabel('examples')
for container in ax.containers:
    ax.bar_label(container, padding=2)
plt.tight_layout()
plt.show()

distribution

## Confusion Matrix

Rows are human labels. Columns are model/rule predictions. The diagonal is correct.

In [ ]:
confusion = pd.crosstab(scores['label'], scores['predicted_label']).reindex(index=LABEL_ORDER, columns=LABEL_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
ax.set_title('Confusion Matrix')
ax.set_xlabel('predicted label')
ax.set_ylabel('actual label')
plt.tight_layout()
plt.show()

confusion

## GumTree Availability

This shows whether GumTree actually produced edit-script features or only fallback values.


In [ ]:
if 'gumtree_available' not in scores.columns:
    print('No GumTree columns found. Run notebook 03 with ENABLE_GUMTREE = True.')
else:
    gumtree_counts = scores['gumtree_available'].value_counts().rename(index={True: 'available', False: 'unavailable'})
    ax = gumtree_counts.plot(kind='bar', figsize=(6, 4), rot=0)
    ax.set_title('GumTree Feature Availability')
    ax.set_xlabel('status')
    ax.set_ylabel('examples')
    for container in ax.containers:
        ax.bar_label(container, padding=2)
    plt.tight_layout()
    plt.show()
    print(gumtree_counts)


## Literal Normalization Impact

This compares normal token recall with literal-tolerant recall. Big gaps mean values changed but surrounding code stayed similar.


In [ ]:
literal_columns = [
    'token_recall',
    'literal_normalized_token_recall',
    'identifier_and_literal_normalized_token_recall',
    'best_hunk_token_recall',
    'best_hunk_literal_normalized_recall',
    'best_hunk_identifier_and_literal_normalized_recall',
]
literal_columns = [column for column in literal_columns if column in scores.columns]

if not literal_columns:
    print('No literal-normalized columns found. Run notebook 03_evaluate again.')
else:
    literal_scores = scores.melt(
        id_vars=['label'],
        value_vars=literal_columns,
        var_name='metric',
        value_name='score',
    )
    grid = sns.catplot(
        data=literal_scores,
        x='label',
        y='score',
        col='metric',
        col_wrap=3,
        kind='box',
        order=LABEL_ORDER,
        height=3.2,
        aspect=1.1,
        sharey=True,
    )
    grid.set_titles('{col_name}')
    grid.set_axis_labels('label', 'score')
    for axis in grid.axes.flatten():
        axis.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()


## Metric Distributions By Human Label

This shows which metrics separate the labels and which ones blur together.

In [ ]:
long_scores = scores.melt(
    id_vars=['example_id', 'label'],
    value_vars=available_metric_columns,
    var_name='metric',
    value_name='score',
)

g = sns.catplot(
    data=long_scores,
    x='label',
    y='score',
    col='metric',
    kind='box',
    order=LABEL_ORDER,
    col_wrap=2,
    height=4,
    sharey=True,
)
g.set_titles('{col_name}')
g.set_axis_labels('human label', 'score')
for ax in g.axes.flat:
    ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

## Token Recall vs Identifier-Normalized Recall

Points above the diagonal usually mean identifier normalization helped, often because variables or helpers were renamed.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(
    data=scores,
    x='token_recall',
    y='identifier_normalized_token_recall',
    hue='label',
    hue_order=LABEL_ORDER,
    style='is_correct',
    alpha=0.75,
    ax=ax,
)
ax.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1)
ax.set_title('Identifier Normalization Gain')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

## Prediction Calibration

A good scoring rule should cluster near the diagonal. Big vertical distance means the percentage estimate is poorly calibrated.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(
    data=scores,
    x='expected_landed_percentage',
    y='predicted_percentage',
    hue='label',
    hue_order=LABEL_ORDER,
    style='is_correct',
    alpha=0.75,
    ax=ax,
)
ax.plot([0, 100], [0, 100], linestyle='--', color='black', linewidth=1)
ax.set_title('Predicted vs Human Percentage')
ax.set_xlabel('human expected landed percentage')
ax.set_ylabel('predicted percentage')
plt.tight_layout()
plt.show()

## Error Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=scores, x='absolute_error', bins=20, hue='label', hue_order=LABEL_ORDER, multiple='stack', ax=ax)
ax.set_title('Absolute Percentage Error')
ax.set_xlabel('absolute error')
plt.tight_layout()
plt.show()

## Anchor Count By Label

If many false positives have zero or very few anchors, the high-score rule is too permissive.


In [ ]:
if 'meaningful_anchor_count' not in scores.columns:
    print('No anchor count column found. Run notebook 03_evaluate again.')
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=scores, x='label', y='meaningful_anchor_count', order=LABEL_ORDER, ax=ax)
    ax.set_title('Meaningful Anchor Count By Human Label')
    ax.set_xlabel('human label')
    ax.set_ylabel('anchor count')
    plt.tight_layout()
    plt.show()


## Biggest Mistakes

In [ ]:
mistake_columns = [
    'example_id',
    'label',
    'predicted_label',
    'expected_landed_percentage',
    'predicted_percentage',
    'absolute_error',
    'line_recall',
    'token_recall',
    'identifier_normalized_token_recall',
    'best_hunk_token_recall',
    'best_hunk_token_precision',
    'best_hunk_token_f1',
    'best_hunk_identifier_normalized_recall',
    'best_hunk_contiguous_line_ratio',
    'best_hunk_token_lcs_recall',
    'best_hunk_size_ratio',
    'meaningful_anchor_recall',
    'meaningful_anchor_count',
    'best_hunk_size',
    'best_hunk_file',
    'file_overlap_ratio',
    'changed_line_overlap_ratio',
]

scores.loc[~scores['is_correct'], mistake_columns].sort_values('absolute_error', ascending=False).head(25)

## False 100% Predictions

These are the most important cases to inspect because the current rule claims the suggestion landed fully.

In [ ]:
scores.loc[
    (scores['predicted_label'] == '100%') & (scores['label'] != '100%'),
    mistake_columns,
].sort_values(['identifier_normalized_token_recall', 'token_recall'], ascending=False).head(25)

## Percentage Buckets

Granular view using `0`, `1-10`, `11-20`, ..., `91-100` buckets.

In [ ]:
bucket_distribution = pd.concat([
    scores['expected_percentage_bucket'].value_counts().rename('expected'),
    scores['predicted_percentage_bucket'].value_counts().rename('predicted'),
], axis=1).reindex(PERCENTAGE_BUCKET_ORDER).fillna(0).astype(int).reset_index(names='bucket')

long_bucket_distribution = bucket_distribution.melt(id_vars='bucket', var_name='series', value_name='rows')
fig, ax = plt.subplots(figsize=(13, 4))
sns.barplot(data=long_bucket_distribution, x='bucket', y='rows', hue='series', ax=ax)
ax.set_title('Expected vs Predicted Percentage Buckets')
ax.set_xlabel('Percentage bucket')
ax.set_ylabel('Rows')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

bucket_distribution

## Bucket Confusion Matrix

In [ ]:
bucket_confusion = pd.crosstab(
    scores['expected_percentage_bucket'],
    scores['predicted_percentage_bucket'],
).reindex(index=PERCENTAGE_BUCKET_ORDER, columns=PERCENTAGE_BUCKET_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(bucket_confusion, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Expected Bucket vs Predicted Bucket')
ax.set_xlabel('Predicted bucket')
ax.set_ylabel('Expected bucket')
plt.tight_layout()
plt.show()

bucket_confusion